# Data exploring

In [ ]:
import pandas as pd

## Check bookings csv

- id is probably connected to the booking_creation_date
- Every product_id always has the same feature_1 value
- Cancellation_date depends on status == 'Cancelled' -> Drop cancellation date because it is always nan when status not cancelled
- Many duplicates -> Drop
- Revenues can be negative due to discounts or other stuff

In [ ]:
bookings = pd.read_csv('../data/20260108_bookings.csv')

In [ ]:
bookings.dtypes

In [ ]:
print(bookings.shape)
bookings.head()

In [ ]:
# Only 238 different product_ids has been booked
# Even if 780 are bookable, according to capacity df
bookings['product_id'].nunique()

### Check feature_1 to product_id correlation

- Yes, correlated

In [ ]:
print(f'Feature 1:\n{bookings["feature_1"].unique()}\n')
print(
    f'Feature 2:\n{bookings["feature_2"].unique()}'
)  # Can be ingnored in data ingestion

In [ ]:
bookings[bookings['feature_1'].isna()]

In [ ]:
bookings[bookings['product_id'] == 'XRJA2U']

In [ ]:
# Feature 1 - Every product_id has only one value for feature_1
count_feature_1_values = bookings.groupby('product_id')['feature_1'].nunique()
count_feature_1_values = count_feature_1_values[count_feature_1_values > 1]
count_feature_1_values

### Check id column

- Does the id belong to the booking_creation_date?
    - Except for one id, every id has only one booking_creation_date

In [ ]:
bookings[(bookings['product_id'] == 'XRJA3') & (bookings['date'] == 20260213)]

In [ ]:
bookings['id'].value_counts()[lambda x: x > 1]
# id columns is not unique?!?!?

In [ ]:
bookings[bookings['id'] == 1093111978486]

In [ ]:
bookings[bookings['id'] == 679335503950].head()

In [ ]:
correlation_id_booking_creation_date = bookings.groupby('id')[
    'booking_creation_date'
].nunique()
correlation_id_booking_creation_date = correlation_id_booking_creation_date[
    correlation_id_booking_creation_date > 1
]
correlation_id_booking_creation_date

# Except for id 636246767275, every id has only one booking_creation_date

In [ ]:
bookings[bookings['id'] == 636246767275]

### Check for duplicated rows

- Yes

In [ ]:
# Check for duplicates
print(len(bookings))
print(len(bookings[bookings.duplicated(keep=False)]))
print(len(bookings[bookings.duplicated(keep='first')]))

len(bookings.drop_duplicates(keep='first'))

### Check date range

In [ ]:
print(bookings['date'].min())
print(bookings['date'].max(), '\n')
# -> Bookings from 2023/04/21 to 2027/04/19

print(bookings['booking_creation_date'].min())
print(bookings['booking_creation_date'].max())
# -> Bookings created from 2018/10/02 to 2025/12/22

### Check cancelled bookings

- cancellation_date is a float
- correlation

In [ ]:
# Cancelled Bookings should be ignored when ingesting data
print(bookings['status'].unique())

print(len(bookings[bookings['status'] == 'Cancelled']) / len(bookings['status']) * 100)
print(len(bookings[bookings['status'] == 'Cancelled']))

In [ ]:
# Get every valid cancellation date and check if the status is Cancelled
df_cancelled_date = bookings[bookings['cancellation_date'].notna()]
print(len(df_cancelled_date))
print(len(df_cancelled_date[df_cancelled_date['status'] == 'Cancelled']))

### Check revenue and currency

In [ ]:
print(
    f'Currencies: {bookings["base_currency"].unique()} \n'
)  # Pfund Sterling (Brit. Pounds) and Euros

print(bookings['gross_revenue'].min())  # Negative values possible
print(bookings['gross_revenue'].max())
print(bookings['gross_revenue'].mean(), '\n')

print(bookings['net_revenue'].min())  # Negative values possible
print(bookings['net_revenue'].max())
print(bookings['net_revenue'].mean())

# bookings[bookings['net_revenue'] == -709.4] # High discount
bookings[bookings['gross_revenue'] == -84.0]  # Looks like a gift :)

## Check capacity csv

- ID represents the product_id and date (bookable unit)
- One line per bookable unit
- 780 different product_ids
- 1702 different dates
- column names (is_...) suggest boolean but are not
- Looks like overbookings are possible

Sum of is_bookable and is_option equals the available units for new customers

In [ ]:
capacity = pd.read_csv('../data/20260108_capacity.csv')

In [ ]:
capacity.dtypes

In [ ]:
print(capacity.shape)
capacity.head()
# One line per "bookable unit"

In [ ]:
capacity[(capacity['product_id'] == 'XRJA3')].sort_values(by='calendar_date')

In [ ]:
print(len(capacity['product_id'].unique()))
# 780 different apartment types
print(len(capacity['calendar_date'].unique()))
# 1702 different dates

print(len(capacity['product_id'].unique()) * len(capacity['calendar_date'].unique()))
# 780 * 1702 = 1327560
# Every product_id has every calendar date listed -> one line per bookable unit

In [ ]:
print(len(capacity['calendar_date'].unique()))
print(capacity['calendar_date'].min())
print(capacity['calendar_date'].max())

In [ ]:
print(f'Is bookable unique values: {capacity["is_bookable"].unique()}\n')
# Count of how many bookable units (product_id + date) are available

print(f'Is option unique values: {capacity["is_option"].unique()}')
# Count of how many bookable units are reservated as optional

# Negativ values = Overbookings?!
# Sum of is_bookable and is_option equals the available units for new customers

## Check prices csv

- ID represents the product_id and date (bookable unit) -> 110.336 unique units
- 264 unique product_ids -> Less then in capacity
- 489 unique dates -> Less then in capacity
    - Not every bookable unit has a price?!
    - Some bookable units have multiple prices?!
    - Maybe because of different length_of_stay but this does not look correct
- length_of_stay is string because of "XN" (Extra Nights?!)
    - Compared to bookings, los has just a small amount available of real bookings

> Every bookable unit has a capacity but not every bookable_unit has a price

> Test hypothese: Every product_id as 552 rows with the same date+length_of_stay pattern


In [ ]:
prices = pd.read_csv('../data/20260108_prices.csv')

In [ ]:
prices.dtypes

In [ ]:
print(prices.shape)
prices.head()

In [ ]:
print(len(prices['id'].unique()))
print(f'Len unique product ids: {len(prices["product_id"].unique())}')
print(f'Len unique dates: {len(prices["date"].unique())}')

In [ ]:
prices[prices['id'] == '65851f3235'].head()
# ID represents the product_id and date (bookable unit)

In [ ]:
df = prices[(prices['product_id'] == 'VISE4D')].sort_values(by='date')
df
# date + length_of_stay = date of next row
# Mayvbe there is a pattern for product_ids in date and length_of_stay

In [ ]:
# Dates
print(f'First date available: {prices["date"].min()}')
print(f'Last data available: {prices["date"].max()}\n')

# Prices
print(f'Min price available: {prices["current_price"].min()}')
print(f'Max price available: {prices["current_price"].max()}\n')

# length_of_stay
print(prices['length_of_stay'].unique())  # XN equals to Extra Night?